# Load Raw BBC Dataset

In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path("../bbc")

rows = []

for label_dir in data_dir.iterdir():
    if label_dir.is_dir():
        label = label_dir.name

        for file_path in label_dir.glob("*.txt"):
            text = file_path.read_text(encoding="latin-1")
            rows.append({
                "text": text,
                "label": label
            })

df = pd.DataFrame(rows)

print(df.head())
print(df["label"].value_counts())

                                                text     label
0  Ad sales boost Time Warner profit\n\nQuarterly...  business
1  Dollar gains on Greenspan speech\n\nThe dollar...  business
2  Yukos unit buyer faces loan claim\n\nThe owner...  business
3  High fuel prices hit BA's profits\n\nBritish A...  business
4  Pernod takeover talk lifts Domecq\n\nShares in...  business
label
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


# Clean text Data

In [2]:
df = df.dropna(subset=["text", "label"])

df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0]

print(df.shape)

(2225, 2)


# Encode Label

In [3]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["label_id"] = label_encoder.fit_transform(df["label"])

print(label_encoder.classes_)
print(df[["label", "label_id"]].head())

['business' 'entertainment' 'politics' 'sport' 'tech']
      label  label_id
0  business         0
1  business         0
2  business         0
3  business         0
4  business         0


# Train/Validation/Test Split

In [4]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label_id"]
)

print(len(train_df), len(val_df), len(test_df))

1780 222 223


# Initialize BertTokenizer

In [5]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

d:\bishoc\AI_vietnam\Project_module1\AIO_NewsClassification\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Testing on 1 text

In [6]:
sample_text = train_df.iloc[0]["text"]

encoding = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

print(encoding.keys())
print(encoding["input_ids"].shape)
print(encoding["attention_mask"].shape)
print(encoding["token_type_ids"].shape)

KeysView({'input_ids': tensor([[  101,  4004,  5085,  9190,  7922,  1005,  1055,  7358,  1996,  7922,
         11842,  2070,  2439,  2598,  2114,  2087,  2350, 12731, 14343, 14767,
          2006,  9317,  2044,  2148,  4420,  1998,  2900,  6380,  2027,  2020,
          4041,  1037,  5271,  1011,  2125,  1012,  1996,  7922,  4265,  2049,
          5221,  2028,  1011,  2154,  2991,  1999,  2176,  2706,  2006,  9857,
          2006, 10069,  2008,  4004,  2430,  5085,  2020,  2055,  2000,  2896,
          2037,  8269,  1997,  6363,  1012,  2900,  2003,  1996,  5221,  9111,
          1997,  7922,  8269,  1999,  1996,  2088,  1010,  2007,  2148,  4420,
          1996,  2959,  2922,  1012,  1996,  7922,  2001,  9343,  9645,  1012,
          6146, 18371,  2012,  5641, 12376, 13938,  2102,  1010,  1014,  1012,
          1019,  1003,  6428,  2006,  1996,  2154,  1012,  2009,  2036, 13011,
          3020,  2114,  2119,  1996,  9944,  1998,  1996,  9044,  1010,  2007,
          2028,  9944,  4276,

In [7]:
input_ids = encoding["input_ids"]
attention_mask = encoding["attention_mask"]
token_type_ids = encoding["token_type_ids"]

# Building Pytorch Dataset

In [8]:
import torch
from torch.utils.data import Dataset

class BBCDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "token_type_ids": encoding["token_type_ids"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# Split Train/Val/Test Dataset

In [9]:
train_dataset = BBCDataset(
    train_df["text"],
    train_df["label_id"],
    tokenizer,
    max_length=256
)

val_dataset = BBCDataset(
    val_df["text"],
    val_df["label_id"],
    tokenizer,
    max_length=256
)

test_dataset = BBCDataset(
    test_df["text"],
    test_df["label_id"],
    tokenizer,
    max_length=256
)

# Buidl DataLoader

In [10]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

# Veify Batch Output

In [11]:
batch = next(iter(train_loader))

print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("token_type_ids:", batch["token_type_ids"].shape)
print("labels:", batch["labels"].shape)

input_ids: torch.Size([16, 256])
attention_mask: torch.Size([16, 256])
token_type_ids: torch.Size([16, 256])
labels: torch.Size([16])


MODEL

In [12]:
import torch
from transformers import BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Mô hình sẽ được huấn luyện trên: {device}")


model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=5
)
model.to(device)  

# 3. Cấu hình các Siêu tham số (Hyperparameters)
epochs = 3
# Learning rate siêu nhỏ (2e-5) để tránh phá vỡ kiến thức cũ của BERT
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

# 4. Thiết lập Scheduler để giảm dần tốc độ học theo thời gian
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,  # Đi thẳng vào learning rate lớn nhất rồi giảm dần
    num_training_steps=total_steps
)

print(f"Tổng số bước huấn luyện (steps): {total_steps}")

Mô hình sẽ được huấn luyện trên: cpu


d:\bishoc\AI_vietnam\Project_module1\AIO_NewsClassification\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6446.15it/s]
[transformers] Bert

Tổng số bước huấn luyện (steps): 336


In [13]:

import numpy as np


def train_epoch(model, data_loader, optimizer, scheduler, device):
    
    model.train()  
    total_train_loss = 0
    
    
    for batch in data_loader:
        
        
        optimizer.zero_grad()
      
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
       
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        total_train_loss += loss.item() 
        

        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # 6. ADAMW VÀ SCHEDULER RA TAY: 
        optimizer.step()   # AdamW thực hiện vặn ốc (cập nhật trọng số) dựa trên bước Backward vừa tính.
        scheduler.step()   # Giảm tốc độ học (Learning Rate) đi một chút xíu theo lộ trình giảm dần đều.
        
    # Trả về sai số trung bình của toàn bộ các mẻ dữ liệu trong vòng này
    return total_train_loss / len(data_loader)


# --- HÀM ĐÁNH GIÁ (VALIDATION) ĐỂ KIỂM TRA NĂNG LỰC ---
def eval_model(model, data_loader, device):
 
    model.eval()  
    total_eval_loss = 0
    correct_predictions = 0
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            total_eval_loss += loss.item()
            
          
            logits = outputs.logits
           
            preds = torch.argmax(logits, dim=1)
            
            
            correct_predictions += torch.sum(preds == labels)
            
    # Tính toán các chỉ số đầu ra
    avg_loss = total_eval_loss / len(data_loader)
    accuracy = correct_predictions.item() / len(data_loader.dataset) # Số câu đúng / Tổng số câu
    
    return avg_loss, accuracy

In [14]:
import time

print("🚀 Kích hoạt vòng lặp chính! Bắt đầu huấn luyện tinh chỉnh BERT...")

# Định cấu hình số vòng lặp (Epochs)
epochs = 3

for epoch in range(epochs):
    # Ghi lại thời gian bắt đầu của Epoch này
    start_time = time.time()
    print(f"\n--- Epoch {epoch + 1} / {epochs} ---")
    
    # BƯỚC 1: Gửi tập TRAIN vào cho mô hình học và cập nhật trọng số
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f"Train Loss (Sai số tập học): {train_loss:.4f}")
    
    # BƯỚC 2: Gửi tập VAL vào để kiểm tra năng lực hiện tại của mô hình
    val_loss, val_acc = eval_model(model, val_loader, device)
    print(f"Validation Loss (Sai số tập thi): {val_loss:.4f}")
    print(f"Validation Accuracy (Độ chính xác thử nghiệm): {val_acc * 100:.2f}%")
    
    # Tính tổng thời gian đã chạy của Epoch này
    elapsed_time = time.time() - start_time
    print(f"Thời gian hoàn thành Epoch: {elapsed_time:.2f} giây")

print("\n✅ Quá trình huấn luyện 3 Epoch hoàn tất!")

🚀 Kích hoạt vòng lặp chính! Bắt đầu huấn luyện tinh chỉnh BERT...

--- Epoch 1 / 3 ---


KeyboardInterrupt: 